# Global Solution 2026 — Análise Interativa
## Monitoramento de Riscos Ambientais com Árvores, Grafos e Algoritmos

**FIAP — Estruturas de Dados e Algoritmos | 1º Semestre 2026**

Este notebook apresenta a análise comparativa dos algoritmos e a escala de decisão.


In [ ]:
import sys, os
sys.path.insert(0, '../src')

import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 100

from data_structures import (
    construir_grafo_rs, construir_grafo_matopiba, construir_bst,
    nome_municipio, risco, custo
)
from brute_force import forca_bruta_caminhos
from greedy import dijkstra, reconstruir_caminho, planejar_atendimento
from performance_monitor import rodar_benchmark_completo, comparar_mesmo_grafo

print('Módulos carregados com sucesso.')

## 1. Cenário A — Grafo RS

In [ ]:
g_rs = construir_grafo_rs()
bst_rs = construir_bst(g_rs)

print(f'Grafo RS: {g_rs}')
print(f'BST RS: {bst_rs}')
print(f'Balanceamento: {bst_rs.balanceamento()}')

print('\nMunicípios por risco crescente:')
for v in bst_rs.percurso_in_order():
    print(f'  {nome_municipio(v):25s}  risco={risco(v):.2f}  custo={custo(v):.0f}')

## 2. Dijkstra — Plano de Atendimento

In [ ]:
hub = 4314902  # Porto Alegre
planos = planejar_atendimento(g_rs, bst_rs, hub, limiar_risco=0.75)

print('\nPlano de atendimento ordenado por prioridade:')
print(f'{"Município":25s} {"Risco":>6} {"Distância":>10}  Caminho')
print('-' * 70)
for p in planos:
    print(f'{p["municipio"]:25s} {p["risco"]:>6.2f} {p["distancia_hub"]:>10.2f}h  '
          f'{"->".join(p["caminho_nomes"])}')

## 3. Força Bruta × Dijkstra — Validação

In [ ]:
import time

origem, destino = 4314902, 4315602  # Porto Alegre → Rio Grande

t0 = time.perf_counter()
res_fb = forca_bruta_caminhos(g_rs, origem, destino)
t_fb = (time.perf_counter() - t0) * 1000

t0 = time.perf_counter()
dist, pred, ops = dijkstra(g_rs, origem)
t_dij = (time.perf_counter() - t0) * 1000

cam_dij = reconstruir_caminho(pred, origem, destino)
nomes_dij = [g_rs.vertices[i][1] for i in cam_dij]
nomes_fb  = [g_rs.vertices[i][1] for i in res_fb['melhor_caminho']]

gap = abs(dist[destino] - res_fb['melhor_custo']) / res_fb['melhor_custo'] * 100 if res_fb['melhor_custo'] > 0 else 0

print(f'Força Bruta:')
print(f'  Caminho: {" → ".join(nomes_fb)}')
print(f'  Custo  : {res_fb["melhor_custo"]:.2f}h')
print(f'  Tempo  : {t_fb:.2f}ms | Caminhos avaliados: {res_fb["avaliados"]:,}')

print(f'\nDijkstra:')
print(f'  Caminho: {" → ".join(nomes_dij)}')
print(f'  Custo  : {dist[destino]:.2f}h')
print(f'  Tempo  : {t_dij:.3f}ms')

print(f'\nGap de otimalidade: {gap:.2f}% ({"ÓTIMO" if gap < 0.001 else "SUBÓTIMO"})')

## 4. Benchmarks e Escala de Decisão

In [ ]:
resultados = rodar_benchmark_completo([5, 8, 10, 12, 20, 50, 100], limite_fb=12)
comparacoes = comparar_mesmo_grafo([4, 5, 6, 7, 8, 9, 10, 11, 12])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Tempo × N
fb  = resultados['forca_bruta']
dij = resultados['dijkstra']
axes[0].plot([r['n'] for r in fb],  [r['tempo_ms'] for r in fb],  'o-r', label='Força Bruta')
axes[0].plot([r['n'] for r in dij], [r['tempo_ms'] for r in dij], 's--b', label='Dijkstra')
axes[0].set_xlabel('N (vértices)'); axes[0].set_ylabel('Tempo (ms)')
axes[0].set_title('Tempo de Execução × N'); axes[0].legend(); axes[0].grid(alpha=0.3)

# Gap %
ns   = [c['n'] for c in comparacoes]
gaps = [c['gap_pct'] for c in comparacoes]
axes[1].bar(ns, gaps, color=['#2ECC71' if g < 0.1 else '#E74C3C' for g in gaps], edgecolor='black')
axes[1].set_xlabel('N (vértices)'); axes[1].set_ylabel('Gap (%)')
axes[1].set_title('Gap FB × Dijkstra (mesmo grafo)'); axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Escala de Decisão

| Nível | Algoritmo | Tamanho N | Qualidade | Custo Computacional | Recomendação |
|-------|-----------|-----------|-----------|--------------------|--------------|
| 1 (Ótimo) | Força Bruta | N ≤ 8 | 100% (global) | Baixo | Validação |
| 2 (Bom) | Dijkstra | N ≤ 100 | ~100% (gap ≈ 0%) | Muito baixo | **Produção** |
| 3 (Aceitável) | Dijkstra heurístico | N ≤ 10.000 | Alto | Baixo | Grande escala |
| 4 (Inviável) | Força Bruta | N > 12 | 100% | Explosão combinatória | Nunca usar |

**Conclusão:** O Dijkstra é a escolha ideal para o cenário real (N=478 municípios do RS),
pois encontra o caminho ótimo (gap = 0% em grafos com pesos não-negativos) com
custo computacional O((V+E) log V) — viável mesmo para centenas de municípios.


In [ ]:
# Cenário B — MATOPIBA
g_mat = construir_grafo_matopiba()
bst_mat = construir_bst(g_mat)

print('Municípios MATOPIBA por risco:')
for v in bst_mat.percurso_in_order():
    print(f'  {nome_municipio(v):25s}  risco={risco(v):.2f}')

hub_mat = 1721000  # Palmas
planos_mat = planejar_atendimento(g_mat, bst_mat, hub_mat, limiar_risco=0.78, orcamento_km=800)

print('\nPlano de atendimento MATOPIBA (orçamento ≤ 800km):')
for p in planos_mat:
    print(f'  {p["municipio"]:25s}  risco={p["risco"]:.2f}  dist={p["distancia_hub"]:.0f}km')